# v7 Counterfactual Ensemble Ranker

This notebook trains the v7 ensemble ranker from `candidates_v7.csv`. It uses outcome-weighted labels, counterfactual positives from losing games, and pairwise within-turn ranking loss.

In [ ]:
from getpass import getpass
import os

if not os.environ.get("HF_TOKEN"):
    token = getpass("HF_TOKEN: ").strip()
    if not token:
        raise RuntimeError("HF_TOKEN is required because this notebook uploads v7 artifacts")
    os.environ["HF_TOKEN"] = token

print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
from pathlib import Path
import os

repo_root = Path.cwd()
if not (repo_root / "notebooks" / "v7" / "train_v7_ranker.py").exists() and (repo_root.parent.parent / "notebooks" / "v7" / "train_v7_ranker.py").exists():
    repo_root = repo_root.parent.parent

data_path = os.environ.get("CANDIDATES_CSV", "").strip()
if data_path:
    print("Using CANDIDATES_CSV:", data_path)
else:
    local = sorted((repo_root / "data").glob("*/candidates_v7.csv"), key=lambda p: p.stat().st_mtime, reverse=True) if (repo_root / "data").exists() else []
    if local:
        os.environ["CANDIDATES_CSV"] = str(local[0])
        print("Using newest local CSV:", local[0])
    else:
        print("No local CSV found. The trainer will try Hugging Face if HF_TOKEN is set.")

In [ ]:
import importlib.util
import sys

if importlib.util.find_spec("torch") is None:
    raise RuntimeError(
        "PyTorch is not installed in this notebook kernel. Run this in a cell, restart the kernel, then rerun training: "
        f"{sys.executable} -m pip install torch"
    )

import torch
print("torch", torch.__version__)

In [ ]:
import os
import subprocess
import sys

cmd = [
    sys.executable,
    "-u",
    str(repo_root / "notebooks" / "v7" / "train_v7_ranker.py"),
    "--export-dir", str(repo_root / "notebooks" / "v7" / "exports"),
    "--epochs", os.environ.get("V7_EPOCHS", "160"),
    "--batch-size", os.environ.get("V7_BATCH_SIZE", "2048"),
    "--pair-loss-weight", os.environ.get("V7_PAIR_LOSS_WEIGHT", "0.80"),
    "--ensemble-size", os.environ.get("V7_ENSEMBLE_SIZE", "4"),
    "--upload",
]
if os.environ.get("CANDIDATES_CSV"):
    cmd.extend(["--csv", os.environ["CANDIDATES_CSV"]])

print("Running:", " ".join(cmd), flush=True)
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
process = subprocess.Popen(
    cmd,
    cwd=str(repo_root),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)
try:
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
finally:
    return_code = process.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, cmd)

In [ ]:
import json

export_dir = repo_root / "notebooks" / "v7" / "exports"
metrics = json.loads((export_dir / "metrics_v7.json").read_text(encoding="utf-8"))
print(json.dumps(metrics, indent=2, sort_keys=True))
print("Model:", export_dir / "model_weights_v7.json")
print("Graphs:", export_dir / "graphs")